In [29]:
import pandas as pd

df = pd.read_csv('db-unza26-csc4792-lusaka_city_council_completed_projects_CLEAN.csv', sep='|', keep_default_na=False)
df['ward'] = 'N/A'
df.to_csv('db-unza26-csc4792-lusaka_city_council_completed_projects_CLEAN.csv', sep='|', index=False, encoding='utf-8')

print("✅ Fixed. ward column now shows 'N/A'")
print(df.to_string())

✅ Fixed. ward column now shows 'N/A'
   project_id              council    city constituency ward                                                                    project_title           project_type     status funding_source                                  source_url extraction_date
0  LCC-COM-01  Lusaka City Council  Lusaka      Kanyama  N/A  CDF gives Twashuka Combined School its first ever double storey classroom block              Education  Completed            CDF  https://www.lcc.gov.zm/completed-projects/      2026-09-13
1  LCC-COM-02  Lusaka City Council  Lusaka      Mandevu  N/A     A newly constructed ablution block at Olympia Market in Mandevu Constituency  Market Infrastructure  Completed            CDF  https://www.lcc.gov.zm/completed-projects/      2026-09-13


In [30]:
import pandas as pd
df = pd.read_csv('db-unza26-csc4792-lusaka_city_council_completed_projects_CLEAN.csv', sep='|')
print("Columns:", list(df.columns))
print("Rows:", len(df))
print(df.to_string())

Columns: ['project_id', 'council', 'city', 'constituency', 'ward', 'project_title', 'project_type', 'status', 'funding_source', 'source_url', 'extraction_date']
Rows: 2
   project_id              council    city constituency  ward                                                                    project_title           project_type     status funding_source                                  source_url extraction_date
0  LCC-COM-01  Lusaka City Council  Lusaka      Kanyama   NaN  CDF gives Twashuka Combined School its first ever double storey classroom block              Education  Completed            CDF  https://www.lcc.gov.zm/completed-projects/      2026-09-13
1  LCC-COM-02  Lusaka City Council  Lusaka      Mandevu   NaN     A newly constructed ablution block at Olympia Market in Mandevu Constituency  Market Infrastructure  Completed            CDF  https://www.lcc.gov.zm/completed-projects/      2026-09-13


In [31]:
import os
print(os.path.exists('lusaka_council_cleaning.ipynb'))
print(os.path.abspath('lusaka_council_cleaning.ipynb'))

True
c:\Users\Admin.ZM-CKTDBC3\lusaka_council_cleaning.ipynb


In [32]:
df.to_csv(CLEAN_FILE, sep='|', index=False, encoding='utf-8')
print(f"✅ CLEAN file saved: {CLEAN_FILE}")
print(f"   Rows: {len(df)}, Columns: {len(df.columns)}")

print("\n=== FINAL FILES ===")
for f in [RAW_BACKUP, CLEAN_FILE]:
    print(f"  {f}  ({os.path.getsize(f)} bytes)")

✅ CLEAN file saved: db-unza26-csc4792-lusaka_city_council_completed_projects_CLEAN.csv
   Rows: 2, Columns: 11

=== FINAL FILES ===
  db-unza26-csc4792-lusaka_city_council_completed_projects_RAW.csv  (468 bytes)
  db-unza26-csc4792-lusaka_city_council_completed_projects_CLEAN.csv  (540 bytes)


In [33]:
print("=== VALIDATION ===")
assert df['project_title'].notna().all(), "Missing project_title"
assert df['constituency'].notna().all(), "Missing constituency"
assert (df['project_title'].str.strip() != '').all(), "Empty project_title"
assert df['project_id'].is_unique, "Duplicate project_ids"
assert not (df['project_title'].str.lower() == 'coming soon').any(), "Placeholders remain"
print("✅ All validation checks passed")

=== VALIDATION ===
✅ All validation checks passed


In [34]:
df = df_raw.copy()

# 1. Remove scraper artifacts: "Coming Soon" placeholder rows
before = len(df)
df = df[df['project_title'].str.strip().str.lower() != 'coming soon'].copy()
print(f"Removed {before - len(df)} placeholder rows")

# 2. Remove duplicates
before = len(df)
df = df.drop_duplicates(subset=['project_title', 'constituency'], keep='first').copy()
print(f"Removed {before - len(df)} duplicate rows")

# 3. Standardise constituency (Title Case, trim)
df['constituency'] = df['constituency'].str.strip().str.title()

# 4. Standardise project_title (trim, collapse spaces)
df['project_title'] = df['project_title'].str.strip().str.replace(r'\s+', ' ', regex=True)

# 5. Standardise status (lowercase then Title Case)
df['status'] = df['status'].str.strip().str.title()

# 6. Derive project_type from title keywords
def classify(title):
    t = title.lower()
    if 'school' in t or 'classroom' in t or 'education' in t: return 'Education'
    if 'market' in t or 'ablution' in t: return 'Market Infrastructure'
    if 'road' in t or 'drainage' in t: return 'Roads & Drainage'
    if 'health' in t or 'clinic' in t: return 'Health'
    if 'water' in t: return 'Water & Sanitation'
    return 'Other'

df['project_type'] = df['project_title'].apply(classify)

# 7. Add schema columns
df['council'] = 'Lusaka City Council'
df['city'] = 'Lusaka'
df['ward'] = 'N/A'  # not published on LCC site
df['funding_source'] = 'CDF'
df['source_url'] = 'https://www.lcc.gov.zm/completed-projects/'
df['extraction_date'] = str(date.today())

# 8. Reorder to common schema
df = df[[
    'project_id', 'council', 'city', 'constituency', 'ward',
    'project_title', 'project_type', 'status',
    'funding_source', 'source_url', 'extraction_date'
]]

# 9. Reset index
df = df.reset_index(drop=True)

print(f"\n✅ CLEAN shape: {df.shape}")
print(df.to_string())

Removed 5 placeholder rows
Removed 0 duplicate rows

✅ CLEAN shape: (2, 11)
   project_id              council    city constituency ward                                                                    project_title           project_type     status funding_source                                  source_url extraction_date
0  LCC-COM-01  Lusaka City Council  Lusaka      Kanyama  N/A  CDF gives Twashuka Combined School its first ever double storey classroom block              Education  Completed            CDF  https://www.lcc.gov.zm/completed-projects/      2026-09-13
1  LCC-COM-02  Lusaka City Council  Lusaka      Mandevu  N/A     A newly constructed ablution block at Olympia Market in Mandevu Constituency  Market Infrastructure  Completed            CDF  https://www.lcc.gov.zm/completed-projects/      2026-09-13


In [35]:
print("=== INSPECTION ===")
print("Columns:", list(df_raw.columns))
print("Rows:", len(df_raw))
print("\nMissing values:")
print(df_raw.isnull().sum())
print("\nDuplicates:", df_raw.duplicated().sum())
print("\n'Coming Soon' placeholders:", (df_raw['project_title'] == 'Coming Soon').sum())
print("\nStatus values:", df_raw['status'].unique())
print("Constituency values:", df_raw['constituency'].unique())

=== INSPECTION ===
Columns: ['project_id', 'constituency', 'project_title', 'status']
Rows: 7

Missing values:
project_id       0
constituency     0
project_title    0
status           0
dtype: int64

Duplicates: 0

'Coming Soon' placeholders: 5

Status values: <StringArray>
['Completed', 'Planned']
Length: 2, dtype: str
Constituency values: <StringArray>
[       'Kanyama',        'Mandevu',        'Chawama',         'Matero',
 'Lusaka Central',         'Munali',        'Kabwata']
Length: 7, dtype: str


In [36]:
import pandas as pd
import shutil
import os
from datetime import date

RAW_FILE = 'db-unza26-csc4792-lusaka_city_council_completed_projects.csv'
RAW_BACKUP = 'db-unza26-csc4792-lusaka_city_council_completed_projects_RAW.csv'
CLEAN_FILE = 'db-unza26-csc4792-lusaka_city_council_completed_projects_CLEAN.csv'

# Preserve RAW copy (unchanged)
shutil.copy(RAW_FILE, RAW_BACKUP)
print(f"✅ RAW backup saved: {RAW_BACKUP}")

# Load RAW
df_raw = pd.read_csv(RAW_BACKUP, sep='|')
print(f"RAW shape: {df_raw.shape}")
print(df_raw.to_string())


✅ RAW backup saved: db-unza26-csc4792-lusaka_city_council_completed_projects_RAW.csv
RAW shape: (7, 4)
   project_id    constituency                                                                    project_title     status
0  LCC-COM-01         Kanyama  CDF gives Twashuka Combined School its first ever double storey classroom block  Completed
1  LCC-COM-02         Mandevu     A newly constructed ablution block at Olympia Market in Mandevu Constituency  Completed
2  LCC-COM-03         Chawama                                                                      Coming Soon    Planned
3  LCC-COM-04          Matero                                                                      Coming Soon    Planned
4  LCC-COM-05  Lusaka Central                                                                      Coming Soon    Planned
5  LCC-COM-06          Munali                                                                      Coming Soon    Planned
6  LCC-COM-07         Kabwata              